# Engenharia de Dados - Projeto E-Commerce (Camada Silver)
**Objetivo:** Processar dados da camada Bronze para Silver, realizando limpeza, padronização, deduplicação sênior e aplicação de regras de negócio para garantir a integridade analítica.

## Configuração do Ambiente
**Objetivo:** Definir o catálogo e o schema silver

In [0]:
# Definindo o catálogo de trabalho
spark.sql("USE CATALOG projeto_medalhao_visagio")

# Garantindo a existência do schema Silver para persistência das tabelas transformadas
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

## Estrutura Padrão por Tabela
**Objetivo:** Tradução de colunas para o português, tipagem correta e a técnica de Deduplicação Sênior via Window Functions para manter apenas o registro mais recente de cada entidade.

### Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType

### `silver.dim_consumidores`
**Origem:** `bronze.tb_customers`

**Transformações:** Tradução de colunas, conversão para Upper Case em momes de Estado e Cidade e Deduplicação Sênior para garantir a unicidade do ID.

In [0]:
# Leitura da Tabela bronze
df_customers_raw = spark.table("bronze.tb_customers")

# Visualização da Estrutura da Tabela bronze
display(df_customers_raw.limit(5))
df_customers_raw.printSchema()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,timestamp_ingestion
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-04-07T16:39:13.990Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-04-07T16:39:13.990Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-04-07T16:39:13.990Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-04-07T16:39:13.990Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-04-07T16:39:13.990Z


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_birth_date: string (nullable = true)
 |-- customer_age: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.window import Window

# Tradução das Colunas e Tipagem
df_customers_transformed = df_customers_raw.select(
    F.col("customer_id").alias("id_consumidor"),
    F.col("customer_unique_id").alias("id_unico_consumidor"),
    F.col("customer_zip_code_prefix").alias("prefixo_cep"),
    F.upper(F.col("customer_city")).alias("cidade"),
    F.upper(F.col("customer_state")).alias("estado"),
    F.col("customer_name").alias("nome_consumidor"),
    F.col("customer_gender").alias("genero_consumidor"),
    F.to_date(F.col("customer_birth_date")).alias("data_nascimento_consumidor"),
    F.col("customer_age").cast(IntegerType()).alias("idade_consumidor"),
    F.col("timestamp_ingestion") 
)

# Definição da Regra de Deduplicação
window_spec = Window.partitionBy("id_consumidor").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Consumidores
df_customers_final = (
    df_customers_transformed
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Escrita no Lakehouse
(df_customers_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_consumidores"))

print("Tabela silver.dim_consumidores carregada e tipada com sucesso.")

Tabela silver.dim_consumidores carregada e tipada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.dim_consumidores").limit(5))

id_consumidor,id_unico_consumidor,prefixo_cep,cidade,estado,nome_consumidor,genero_consumidor,data_nascimento_consumidor,idade_consumidor
00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,6273,OSASCO,SP,Dr. Davi Pinto,M,1971-07-03,55
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,ITAPECERICA,MG,Sr. Ravi Lucca Sousa,M,1991-03-07,35
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,NOVA VENECIA,ES,Giovanna Ramos,F,1952-02-19,74
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,MENDONCA,MG,Lívia Nogueira,F,2008-02-05,18
000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,4841,SAO PAULO,SP,Srta. Maria Laura Moura,F,1952-06-29,74


### `silver.fat_pedidos`
**Origem:** `bronze.tb_orders`

**Transformações:** Tradução de status para português, conversão de datas com try_to_timestamp e criação de KPIs de logística (tempo de entrega real vs. estimado).

In [0]:
# Leitura da Tabela bronze
df_orders_raw = spark.table("bronze.tb_orders")

# Visualização da Estrutura da Tabela bronze
display(df_orders_raw.limit(5))
df_orders_raw.printSchema()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timestamp_ingestion
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-04-07T16:39:48.186Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-04-07T16:39:48.186Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-04-07T16:39:48.186Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-04-07T16:39:48.186Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-04-07T16:39:48.186Z


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType

# Mapeamento de Status (Regra de Negócio: Inglês -> Português)
status_mapping = {
    "delivered": "entregue", "canceled": "cancelado", "shipped": "enviado",
    "processing": "processando", "invoiced": "faturado", "unavailable": "indisponível",
    "created": "criado", "approved": "aprovado"
}

# Criação da expressão condicional para tradução
status_expr = F.create_map([F.lit(x) for x in [val for pair in status_mapping.items() for val in pair]])

# Tradução das Colunas e Tipagem
df_orders_transformed = df_orders_raw.select(
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    status_expr[F.col("order_status")].alias("status"),
    F.try_to_timestamp(F.col("order_purchase_timestamp")).alias("data_pedido"),
    F.try_to_timestamp(F.col("order_approved_at")).alias("data_aprovacao"),
    F.try_to_timestamp(F.col("order_delivered_carrier_date")).alias("data_entrega_transportadora"),
    F.try_to_timestamp(F.col("order_delivered_customer_date")).alias("data_entrega_real"),
    F.try_to_timestamp(F.col("order_estimated_delivery_date")).alias("data_entrega_estimada"),
    F.col("timestamp_ingestion")
)

# Criação de Colunas Calculadas (KPIs de Entrega)
# Foi orientado no grupo a usar "entrega_no_prazo" baseada em diferenca_entrega_dias e  order_delivered_customer_date
df_orders_with_metrics = df_orders_transformed.withColumn(
    "tempo_entrega_dias", 
    F.datediff(F.col("data_entrega_real"), F.col("data_pedido"))
).withColumn(
    "tempo_entrega_estimado_dias", 
    F.datediff(F.col("data_entrega_estimada"), F.col("data_pedido"))
).withColumn(
    "diferenca_entrega_dias", 
    F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
).withColumn(
    "entrega_no_prazo",
    F.when(F.col("data_entrega_real").isNull(), "Não Entregue")
     .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
     .otherwise("Não")
)

# # Escrita no Lakehouse
(df_orders_with_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedidos"))

print("Tabela silver.fat_pedidos processada com KPIs de entrega.")

Tabela silver.fat_pedidos processada com KPIs de entrega.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.fat_pedidos").limit(5))

id_pedido,id_consumidor,status,data_pedido,data_aprovacao,data_entrega_transportadora,data_entrega_real,data_entrega_estimada,timestamp_ingestion,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-04-07T16:39:48.186Z,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-04-07T16:39:48.186Z,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-04-07T16:39:48.186Z,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-04-07T16:39:48.186Z,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-04-07T16:39:48.186Z,3,13,-10,Sim


### `silver.fat_itens_pedidos`
**Origem:** `bronze.tb_order_items`

**Transformações:** Tradução de colunas e tipagem de valores financeiros com arredondamento de duas casas decimais. Aplicação de Deduplicação Sênior pela chave composta de pedido e item para garantir a integridade dos dados.

In [0]:
# Leitura da Tabela bronze
df_order_items_raw = spark.table("bronze.tb_order_items")

# Visualização da Estrutura da Tabela bronze
display(df_order_items_raw.limit(5))
df_order_items_raw.printSchema()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,timestamp_ingestion
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2026-04-07T16:39:32.977Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2026-04-07T16:39:32.977Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2026-04-07T16:39:32.977Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2026-04-07T16:39:32.977Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2026-04-07T16:39:32.977Z


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql.types import StringType, IntegerType, DoubleType

# Tradução das Colunas e Tipagem
df_items_transformed = df_order_items_raw.select(
    F.col("order_id").alias("id_pedido"),
    F.col("order_item_id").alias("id_item"),
    F.col("product_id").alias("id_produto"),
    F.col("seller_id").alias("id_vendedor"),
    F.try_to_timestamp(F.col("shipping_limit_date")).alias("data_limite_envio"),
    F.round(F.col("price").cast(DoubleType()), 2).alias("preco_BRL"),
    F.round(F.col("freight_value").cast(DoubleType()), 2).alias("preco_frete"),
    
    F.col("timestamp_ingestion")
)

# Definição da Regra de Deduplicação
window_spec = Window.partitionBy("id_pedido", "id_item").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Itens Pedidos
df_items_final = (
    df_items_transformed
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_items_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_itens_pedidos"))

print("Tabela silver.fat_itens_pedidos carregada com sucesso.")

Tabela silver.fat_itens_pedidos carregada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.fat_itens_pedidos").limit(5))

id_pedido,id_item,id_produto,id_vendedor,data_limite_envio,preco_BRL,preco_frete
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14


### `silver.fat_pagamentos_pedidos`
**Origem:** `bronze.tb_order_payments`

**Transformações:** Tradução de tipos de pagamento para português, tipagem numérica com arredondamento financeiro e deduplicação baseada na chave composta de pedido e sequência de pagamento.

In [0]:
# Leitura da Tabela bronze
df_payments_raw = spark.table("bronze.tb_order_payments")

# Visualização da Estrutura da Tabela bronze
display(df_payments_raw.limit(5))
df_payments_raw.printSchema()

order_id,payment_sequential,payment_type,payment_installments,payment_value,timestamp_ingestion
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-04-07T16:39:38.354Z
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-04-07T16:39:38.354Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-04-07T16:39:38.354Z
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-04-07T16:39:38.354Z
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-04-07T16:39:38.354Z


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType

# Mapeamento de Tipos de Pagamento (Regra: Inglês -> Português)
pagamento_mapping = {
    "credit_card": "Cartão de Crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de Débito",
    "not_defined": "Não Definido"
}
pagamento_expr = F.create_map([F.lit(x) for x in [val for pair in pagamento_mapping.items() for val in pair]])

# Tradução das Colunas e Tipagem
df_payments_transformed = df_payments_raw.select(
    F.col("order_id").alias("id_pedido"),
    F.col("payment_sequential").cast(IntegerType()).alias("sequencial_pagamento"),
    pagamento_expr[F.col("payment_type")].alias("tipo_pagamento"),
    F.col("payment_installments").cast(IntegerType()).alias("parcelas_pagamento"),
    F.round(F.col("payment_value").cast(DoubleType()), 2).alias("valor_pagamento_BRL"),
    F.col("timestamp_ingestion")
)

# Definição da Regra de Deduplicação
# A chave de unicidade aqui é o id_pedido + sequencial_pagamento
window_spec = Window.partitionBy("id_pedido", "sequencial_pagamento").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Pagamentos dos Pedidos
df_payments_final = (
    df_payments_transformed
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_payments_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pagamentos_pedidos"))

print("Tabela silver.fat_pagamentos_pedidos carregada com sucesso.")

Tabela silver.fat_pagamentos_pedidos carregada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.fat_pagamentos_pedidos").limit(5))

id_pedido,sequencial_pagamento,tipo_pagamento,parcelas_pagamento,valor_pagamento_BRL
00010242fe8c5a6d1ba2dd792cb16214,1,Cartão de Crédito,2,72.19
00018f77f2f0320c557190d7a144bdd3,1,Cartão de Crédito,3,259.83
000229ec398224ef6ca0657da4fc703e,1,Cartão de Crédito,5,216.87
00024acbcdf0a6daa1e931b038114c75,1,Cartão de Crédito,2,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,1,Cartão de Crédito,3,218.04


### `silver.fat_avaliacoes_pedidos`
**Origem:** `bronze.tb_order_reviews`

**Transformações:** Remoção de registros inválidos ou futuros, preenchimento de nulos com textos padrão e tradução de colunas com suporte a falhas via try_to_timestamp.

In [0]:
# Leitura da Tabela bronze
df_reviews_raw = spark.table("bronze.tb_order_reviews")

# Visualização da Estrutura da Tabela bronze
display(df_reviews_raw.limit(5))
df_reviews_raw.printSchema()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,timestamp_ingestion
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-04-07T16:39:43.048Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-04-07T16:39:43.048Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-04-07T16:39:43.048Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-04-07T16:39:43.048Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-04-07T16:39:43.048Z


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType

# Tradução das Colunas, Tipagem e Tratamento de Nulos
df_reviews_transformed = df_reviews_raw.select(
    F.col("review_id").alias("id_avaliacao"),
    F.col("order_id").alias("id_pedido"),
    F.expr("try_cast(review_score as int)").alias("nota_avaliacao"),
    F.coalesce(F.col("review_comment_title"), F.lit("Sem título")).alias("titulo_avaliacao"),
    F.coalesce(F.col("review_comment_message"), F.lit("Sem comentário")).alias("comentario_avaliacao"),
    F.try_to_timestamp(F.col("review_creation_date")).alias("data_criacao_avaliacao"),
    F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao"),
    F.col("timestamp_ingestion")
)

# Filtros de Qualidade
df_reviews_filtered = df_reviews_transformed.filter(
    (F.col("id_pedido").isNotNull()) & 
    (F.col("data_criacao_avaliacao") <= F.current_timestamp())
)

# Definição da Regra de Deduplicação
# Garante a avaliação mais recente caso haja duplicatas de id_avaliacao
window_spec = Window.partitionBy("id_avaliacao").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Avaliações dos Pedidos
df_reviews_final = (
    df_reviews_filtered
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_reviews_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_avaliacoes_pedidos"))

print("Tabela silver.fat_avaliacoes_pedidos processada com sucesso.")

Tabela silver.fat_avaliacoes_pedidos processada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.fat_avaliacoes_pedidos").limit(5))

id_avaliacao,id_pedido,nota_avaliacao,titulo_avaliacao,comentario_avaliacao,data_criacao_avaliacao,data_resposta_avaliacao
null,material de boa qualidade.Agora é amarelo mesmo,null,"mas e muito bonito.eu recomendo""",2018-01-06 00:00:00,2018-01-08T14:20:31.000Z,null
Ao se colocar o Celular,na CAPA,null,o Microfone,"ficam Bloquados ......e sem Acess""",2018-04-04T00:00:00.000Z,2018-04-06T16:36:40.000Z
0001239bc1de2e33cb583967c2ca4c67,fc046d7776171871436844218f817d7d,5,Sem título,Sem comentário,2018-03-20T00:00:00.000Z,2018-03-20T18:36:04.000Z
0001cc6860aeaf5b9017fe4131a52e62,d4665434b01caa9dc3e3e78b3eb3593e,5,Sem título,Sem comentário,2018-06-22T00:00:00.000Z,2018-06-26T13:51:29.000Z
00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,2018-04-25T00:00:00.000Z,2018-04-26T14:55:36.000Z


### `silver.dim_produtos`
**Origem:** `bronze.tb_products`

**Transformações:** Tradução de colunas para português, tipagem numérica de dimensões e pesos, e aplicação de Deduplicação Sênior via Window Function para garantir a unicidade do produto.

In [0]:
# Leitura da Tabela bronze
df_products_raw = spark.table("bronze.tb_products")

# Visualização da Estrutura da Tabela bronze
display(df_products_raw.limit(5))
df_products_raw.printSchema()

product_id,product_name,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,timestamp_ingestion
1e9e8ef04dbcff4541ed26657ea517e5,Perfume Premium,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,2026-04-07T16:39:54.982Z
3aa071139cb16b67ca9e5dea641aaa2f,Conjunto de Pincéis,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,2026-04-07T16:39:54.982Z
96bd76ec8810374ed1b65e291975717f,Barraca de Camping,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,2026-04-07T16:39:54.982Z
cef67bcfe19066a932b7673e239eb23d,Chupeta Premium,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,2026-04-07T16:39:54.982Z
9dc1a7de274444849c219cff195d0b71,Vassoura Mágica,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,2026-04-07T16:39:54.982Z


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType

# Tradução das Colunas e Tipagem
df_products_transformed = df_products_raw.select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("product_category_name").alias("categoria_produto"),
    F.col("product_weight_g").cast(DoubleType()).alias("peso_produto_gramas"),
    F.col("product_length_cm").cast(DoubleType()).alias("comprimento_centimetros"),
    F.col("product_height_cm").cast(DoubleType()).alias("altura_centimetros"),
    F.col("product_width_cm").cast(DoubleType()).alias("largura_centimetros"),
    F.col("product_photos_qty").cast(DoubleType()).cast(IntegerType()).alias("quantidade_fotos"),
    F.col("product_name_lenght").cast(DoubleType()).cast(IntegerType()).alias("tamanho_nome_produto"),
    F.col("product_description_lenght").cast(DoubleType()).cast(IntegerType()).alias("tamanho_descricao_produto"),
    F.col("timestamp_ingestion")
)

# Definição da Regra de Deduplicação
window_spec = Window.partitionBy("id_produto").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Produtos
df_products_final = (
    df_products_transformed
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_products_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_produtos"))

print("Tabela silver.dim_produtos carregada com sucesso!")

Tabela silver.dim_produtos carregada com sucesso!


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.dim_produtos").limit(5))

id_produto,nome_produto,categoria_produto,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,quantidade_fotos,tamanho_nome_produto,tamanho_descricao_produto
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,perfumaria,300.0,20.0,16.0,16.0,6,53,596
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,automotivo,1225.0,55.0,10.0,26.0,4,56,752
0009406fd7479715e4bef61dd91f2462,Toalha de Banho Premium,cama_mesa_banho,300.0,45.0,15.0,35.0,2,50,266
000b8f95fcb9e0096488278317764d19,Tábua de Corte,utilidades_domesticas,550.0,19.0,24.0,12.0,3,25,364
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,relogios_presentes,250.0,22.0,11.0,15.0,4,48,613


### `silver.dim_vendedores`
**Origem:** `bronze.tb_sellers`

**Transformações:** Tradução de colunas, conversão de Cidade e Estado para Upper Case e aplicação de Deduplicação Sênior para garantir a unicidade do vendedor via Window Function

In [0]:
# Leitura da Tabela bronze
df_sellers_raw = spark.table("bronze.tb_sellers")

# Visualização da Estrutura da Tabela bronze
display(df_sellers_raw.limit(5))
df_sellers_raw.printSchema()

seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_name,seller_registration_date,timestamp_ingestion
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,Dr. Pedro Miguel Vasconcelos,2018-06-25,2026-04-07T16:39:58.855Z
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,Gabriela da Cruz,2018-09-21,2026-04-07T16:39:58.855Z
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,Luigi Viana,2018-04-03,2026-04-07T16:39:58.855Z
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,Beatriz Brito,2018-12-20,2026-04-07T16:39:58.855Z
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,Gabrielly Pastor,2017-08-24,2026-04-07T16:39:58.855Z


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_registration_date: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

# Tradução das Colunas e Tipagem
df_sellers_transformed = df_sellers_raw.select(
    F.col("seller_id").alias("id_vendedor"),
    F.col("seller_name").alias("nome_vendedor"),
    F.col("seller_zip_code_prefix").alias("prefixo_cep"),
    F.upper(F.col("seller_city")).alias("cidade"),
    F.upper(F.col("seller_state")).alias("estado"),
    F.to_date(F.col("seller_registration_date")).alias("seller_registration_date"),
    F.col("timestamp_ingestion")
)

# Definição da Regra de Deduplicação
# Mesma regra de Window Function baseada no timestamp_ingestion
window_spec = Window.partitionBy("id_vendedor").orderBy(F.col("timestamp_ingestion").desc())

# Aplicando a Deduplicação de Vendedores
df_sellers_final = (
    df_sellers_transformed
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_sellers_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_vendedores"))

print("Tabela silver.dim_vendedores processada com sucesso.")

Tabela silver.dim_vendedores processada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.dim_vendedores").limit(5))

id_vendedor,nome_vendedor,prefixo_cep,cidade,estado,seller_registration_date
0015a82c2db000af6aaaf3ae2ecb0532,Amanda Sá,9080,SANTO ANDRE,SP,2018-09-29
001cca7ae9ae17fb1caed9dfb1094831,Vinicius Nogueira,29156,CARIACICA,ES,2017-11-09
001e6ad469a905060d959994f1b41e4f,Ana Clara Moreira,24754,SAO GONCALO,RJ,2018-11-16
002100f778ceb8431b7a1020ff7ab48f,Srta. Emanuella Rezende,14405,FRANCA,SP,2017-04-17
003554e2dce176b5555353e4f3555ac8,Rebeca Costa,74565,GOIANIA,GO,2017-07-20


### `silver.dim_categoria_produtos_traducao`
**Origem:** `bronze.tb_product_category_name_translation`

**Transformações:** Mapeamento de nomes de categorias de inglês para português e aplicação de Deduplicação Sênior para garantir a unicidade de cada termo traduzido.

In [0]:
# Leitura da Tabela bronze
df_category_raw = spark.table("bronze.tb_product_category_name_translation")

# Visualização da Estrutura da Tabela bronze
display(df_category_raw.limit(5))
df_category_raw.printSchema()

product_category_name,product_category_name_english,timestamp_ingestion
beleza_saude,health_beauty,2026-04-07T16:40:03.036Z
informatica_acessorios,computers_accessories,2026-04-07T16:40:03.036Z
automotivo,auto,2026-04-07T16:40:03.036Z
cama_mesa_banho,bed_bath_table,2026-04-07T16:40:03.036Z
moveis_decoracao,furniture_decor,2026-04-07T16:40:03.036Z


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window



# Tradução das Colunas
df_category_transformed = df_category_raw.select(
    F.col("product_category_name").alias("nome_produto_pt"),
    F.col("product_category_name_english").alias("nome_produto_en"),
    F.col("timestamp_ingestion")
)

# Definição da Regra de Deduplicação
# Garante que, se houver múltiplas cargas da mesma categoria, pegamos a última
window_spec = Window.partitionBy("nome_produto_pt").orderBy(F.col("timestamp_ingestion").desc())  # VER SE FAZ SENTIDO

# Aplicando a Deduplicação de Vendedores
df_category_final = (
    df_category_transformed
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "timestamp_ingestion")
)

# Gravando na Camada Silver
(df_category_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_categoria_produtos_traducao"))

print("Tabela silver.dim_categoria_produtos_traducao criada com sucesso conforme mapeamento.")

Tabela silver.dim_categoria_produtos_traducao criada com sucesso conforme mapeamento.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.dim_categoria_produtos_traducao").limit(5))

nome_produto_pt,nome_produto_en
agro_industria_e_comercio,agro_industry_and_commerce
alimentos,food
alimentos_bebidas,food_drink
artes,art
artes_e_artesanato,arts_and_craftmanship


### `silver.dim_cotacao_dolar`
**Origem:** `bronze.tb_cotacao_dolar`

**Transformações:** Criação de calendário contínuo para suprir lacunas de finais de semana e preenchimento de valores nulos utilizando a cotação do último dia útil anterior via Window Function.

In [0]:
# Leitura da Tabela bronze
df_cotacao_raw = spark.table("bronze.tb_cotacao_dolar")

# Visualização da Estrutura da Tabela bronze
display(df_cotacao_raw.limit(5))
df_cotacao_raw.printSchema()

cotacaoCompra,dataHoraCotacao,timestamp_ingestion
3.2715,2016-09-05 13:09:55.659,2026-04-07T16:40:07.312Z
3.2446,2016-09-06 13:02:39.984,2026-04-07T16:40:07.312Z
3.1928,2016-09-08 13:03:53.968,2026-04-07T16:40:07.312Z
3.2632,2016-09-09 13:14:00.885,2026-04-07T16:40:07.312Z
3.2848,2016-09-12 13:08:01.541,2026-04-07T16:40:07.312Z


root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

# Obtendo a menor e maior data para criar um calendário contínuo
# Garante cobertura completa do período retornado pela API
range_dates = df_cotacao_raw.select(
    F.min(F.to_date("dataHoraCotacao")).alias("min_date"),
    F.max(F.to_date("dataHoraCotacao")).alias("max_date")
).collect()[0]

# Cria um calendário diário entre min_date e max_date
df_calendario = spark.range(1).select(
    F.explode(F.sequence(
        F.lit(range_dates['min_date']), 
        F.lit(range_dates['max_date']), 
        F.expr("interval 1 day")
    )).alias("data_referencia")
)

# Converte timestamp para data para permitir join diário
df_bronze_prep = df_cotacao_raw.withColumn("data_cotacao_base", F.to_date("dataHoraCotacao"))

# Join com calendário para garantir todas as datas
df_joined = df_calendario.join(
    df_bronze_prep, 
    df_calendario.data_referencia == df_bronze_prep.data_cotacao_base, 
    "left"
)

# Janela acumulativa ordenada por data
# Usada para preencher valores faltantes com último válido
window_spec = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)


# Preenche datas sem cotação com último valor disponível
# Arredonda para 4 casas e define schema final
df_cotacao_final = df_joined.withColumn(
    "valor_cotacao_dolar", 
    F.last("cotacaoCompra", ignorenulls=True).over(window_spec)
).select(
    F.col("data_referencia").alias("data_cotacao"),
    F.round(F.col("valor_cotacao_dolar").cast(DoubleType()), 2).alias("valor_cotacao_dolar")
)

# Gravando na Camada Silver
(df_cotacao_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_cotacao_dolar"))

print("Tabela silver.dim_cotacao_dolar processada com sucesso.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.dim_cotacao_dolar processada com sucesso.


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.dim_cotacao_dolar").limit(5))

data_cotacao,valor_cotacao_dolar
2016-09-05,3.27
2016-09-06,3.24
2016-09-07,3.24
2016-09-08,3.19
2016-09-09,3.26


## Tabela Final da Silver
**Objetivo:** Consolidar dados de pedidos e pagamentos em uma visão única, aplicando a conversão monetária para USD e garantindo o arredondamento de duas casas decimais para análise financeira.

### `silver.fat_pedido_total`
**Descrição:** Consolidar a visão final de vendas na camada Silver, unificando dados de pedidos, pagamentos agregados e conversão cambial diária

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Leitura das tabelas Silver
df_pedidos = spark.table("silver.fat_pedidos")
df_pagamentos = spark.table("silver.fat_pagamentos_pedidos")
df_cotacao = spark.table("silver.dim_cotacao_dolar")

# Agregação de Pagamentos por Pedido
# Soma-se os pagamentos para obter o valor total por pedido
df_pagamentos_agg = df_pagamentos.groupBy("id_pedido").agg(
    F.sum("valor_pagamento_brl").alias("total_brl")
)

# Join Final (Pedidos + Pagamentos + Cotação)
# O join com a cotação usa a data do pedido (sem a hora)
df_final = (
    df_pedidos.alias("p")
    .join(df_pagamentos_agg.alias("pag"), "id_pedido", "inner")
    .join(df_cotacao.alias("c"), F.to_date(F.col("p.data_pedido")) == F.col("c.data_cotacao"), "left")
)

# Seleção de Colunas e Arredondamento Obrigatório (2 casas)
df_fat_pedido_total = df_final.select(
    F.col("p.id_pedido"),
    F.col("p.id_consumidor"),
    F.col("p.status"),
    F.round(F.col("pag.total_brl"), 2).alias("valor_total_pago_brl"),
    F.round(F.col("pag.total_brl") / F.col("c.valor_cotacao_dolar"), 2).alias("valor_total_pago_usd"),
    F.col("p.data_pedido")
)

# Gravação da Tabela Fato Final
(df_fat_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedido_total"))

# Otimização Física
# Indexação por id_pedido e data_pedido para performance analítica
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

print("Tabela silver.fat_pedido_total criada e otimizada com sucesso!")

Tabela silver.fat_pedido_total criada e otimizada com sucesso!


In [0]:
# Validação da Estrutura da Tabela silver
display(spark.table("silver.fat_pedido_total").limit(50))

id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,38.71,12.25,2017-10-02T10:56:33.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,141.46,37.72,2018-07-24T20:41:37.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,179.12,47.77,2018-08-08T08:38:49.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,72.2,22.01,2017-11-18T19:28:06.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,28.62,8.73,2018-02-13T21:18:39.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,entregue,175.26,53.27,2017-07-09T21:57:05.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,faturado,65.95,21.0,2017-04-11T12:22:08.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,entregue,75.16,24.32,2017-05-16T13:10:30.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,entregue,35.95,11.38,2017-01-23T18:29:09.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,entregue,169.76,53.89,2017-07-29T11:55:02.000Z
